## ANALISIS CROSS SELLING PARA CAMPAÑAS
### CÁLCULO DE CORRELACION ENTRE PARES SKUs 
Autor: Flavia Davila   
2026

In [ ]:
# Importamos librerias necesarias
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter, defaultdict

### Load the data

In [4]:
# Leemos la hoja armada en base al cubo para extraer el arbol de categorizacion
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [5]:
# leemos el csv completo de ventas facturadas del periodo 
ventasune_farma = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\scripts\febrero_2026.csv")

In [7]:
# leemos informacion de los clusters de la UNE
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

### PROCESAMOS DATA

In [15]:
#convertimos fechas a formato datetime en caso de querer filtrar periodos especificos
ventasune_farma['FECHA_FACT']= pd.to_datetime(ventasune_farma['FECHA_FACT'])

In [16]:
#analizamos el periodo de interés
#ventasxfact_farma = ventasune_farma[(ventasune_farma['FECHA_FACT'] >= '2025-03-01 00:00:00') & (ventasune_farma['FECHA_FACT'] <= '2025-03-15 00:00:00')]
ventasxfact_farma = ventasune_farma.copy()

In [17]:
# visualizamos el formato de la data
ventasxfact_farma.head()

,FECHA_FACT,COD_BODEGA,NUMERO_FACTURA,COD_ARTICULO,UNIDADES,VENTA_NETA
0,2026-02-27,B701,2181484B1,330617,1.0,170.7
1,2026-02-25,B224,275510CM,330617,1.0,170.7
2,2026-02-24,B703,710882B3,330617,1.0,170.7
3,2026-02-22,B1PA,488617SCJ,330617,1.0,170.7
4,2026-02-21,B303,1725946L3,330617,1.0,170.7


In [18]:
# Agrupamos para contar cantidad de facturas emitidas por dia en el periodo analizado
ventasfactxdia = (
    ventasxfact_farma
    .groupby('FECHA_FACT')['NUMERO_FACTURA']
    .nunique()
    .reset_index(name='FACTURAS_UNICAS')
)


In [14]:
# guardamos la cantidad de facturas unicas emitidas en el periodo analizado
total_fact_periodo = ventasfactxdia['FACTURAS_UNICAS'].sum()
total_fact_periodo

np.int64(1978143)

In [19]:
# filtramos solo BODEGAS de la UNE de interés
#clusters_amkt = clusters[clusters['UNE']=='AMARKET']
clusters_farma = clusters[clusters['UNE']=='FARMACORP']

## CROSS SELLING X PERIODO

### Procesamos la data

In [20]:
# convertimos los IDs en string
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact_farma['COD_ARTICULO'] = ventasxfact_farma['COD_ARTICULO'].astype(str)

In [23]:
# unimos la informacion de los SKUs con el conteo de ventasxfactura
ventas_farma = pd.merge(cubo,
                        ventasxfact_farma,
                        on='COD_ARTICULO',
                        how='inner'
                        )

In [24]:
# traemos la info de los clusters 
ventas_farmacorp = pd.merge(
    ventas_farma, 
    clusters_farma,
    left_on='COD_BODEGA',
    right_on ='BODEGA',
    how='left')

### Apertura por REGION

In [26]:
ventas_farmacorp.CIUDAD.value_counts()

CIUDAD
SANTA CRUZ    2031391
LA PAZ         348415
COCHABAMBA     303426
TARIJA         156505
BENI            74248
ORURO           59030
PANDO           45261
SUCRE           33527
POTOSI          18987
Name: count, dtype: int64

In [27]:
# Filtramos por ciudad ('COCHABAMBA', 'LA PAZ', 'TARIJA', 'BENI', 'ORURO', 'PANDO', 'SUCRE', 'POTOSI')
#ciudad = 'SANTA CRUZ' 
#ventas_region_farma = ventas_farmacorp[ventas_farmacorp['CIUDAD']==ciudad]

# Todas las ventas
ventas_region_farma = ventas_farmacorp.copy()

In [29]:
# Calculamos pares de SKUs frecuencia de ocurrencia conjunta y moda de ocurrencias.
pares = Counter()
unidades_pares = defaultdict(Counter)

cat1_map = ventas_region_farma.set_index('ARTICULO')['CAT 2'].to_dict()

ventas_sorted = ventas_region_farma[['NUMERO_FACTURA', 'ARTICULO', 'UNIDADES']].sort_values('NUMERO_FACTURA')

factura_actual = None
articulos_actuales = []

for row in ventas_sorted.itertuples(index=False):
    if row.NUMERO_FACTURA != factura_actual:
        if len(articulos_actuales) >= 2:

            articulos_ordenados = sorted(articulos_actuales, key=lambda x: x[0])
            for (art_a, und_a), (art_b, und_b) in combinations(articulos_ordenados, 2):
                if cat1_map.get(art_a) != cat1_map.get(art_b):

                    pares[(art_a, art_b)] += 1
                    unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1
                    
        factura_actual = row.NUMERO_FACTURA
        articulos_actuales = [(row.ARTICULO, row.UNIDADES)]
    else:
        articulos_actuales.append((row.ARTICULO, row.UNIDADES))

if len(articulos_actuales) >= 2:
    articulos_ordenados = sorted(articulos_actuales, key=lambda x: x[0])
    for (art_a, und_a), (art_b, und_b) in combinations(articulos_ordenados, 2):
        if cat1_map.get(art_a) != cat1_map.get(art_b):
            pares[(art_a, art_b)] += 1
            unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1


resultados_pares = []
for (a, b), freq in pares.items():
    (moda_und_a, moda_und_b), frec_moda = unidades_pares[(a, b)].most_common(1)[0]
    
    resultados_pares.append({
        'ARTICULO_A': a,
        'ARTICULO_B': b,
        'FRECUENCIA': freq,
        'MODA_UNIDADES_A': moda_und_a,
        'MODA_UNIDADES_B': moda_und_b,
        'FREC_MODA_UNIDADES': frec_moda
    })

df_pares = pd.DataFrame(resultados_pares)

cat_cols = ['ARTICULO', 'CAT 1', 'CAT 2', 'CAT 3', 'CAT 4']
attrs = ventas_region_farma.drop_duplicates('ARTICULO').set_index('ARTICULO')[cat_cols[1:]]

df_pares = df_pares.join(attrs.add_suffix('_A'), on='ARTICULO_A')
df_pares = df_pares.join(attrs.add_suffix('_B'), on='ARTICULO_B')

df_pares = df_pares.sort_values('FRECUENCIA', ascending=False).reset_index(drop=True)

In [30]:
# Calculamos cuántas facturas únicas tiene cada artículo por separado
frecuencias_individuales = ventas_region_farma.groupby('ARTICULO')['NUMERO_FACTURA'].nunique().to_dict()

# Mapeamos esas frecuencias a los artículos A y B en nuestro DataFrame de pares
df_pares['FREQ_A'] = df_pares['ARTICULO_A'].map(frecuencias_individuales)
df_pares['FREQ_B'] = df_pares['ARTICULO_B'].map(frecuencias_individuales)

# Calculamos la correlación (Confianza) para A y para B
df_pares['CONF_A'] = df_pares['FRECUENCIA'] / df_pares['FREQ_A']
df_pares['CONF_B'] = df_pares['FRECUENCIA'] / df_pares['FREQ_B']


In [31]:
# copiamos el DF
pairs_farma_region = df_pares.copy()

In [34]:
# Calculamos la tasa de frecuencia en relacion al total de facturas emitidas
pairs_farma_region['TASA_AB'] = pairs_farma_region['FRECUENCIA']/ total_fact_periodo

In [36]:
pairs_2025 = pairs_farma_region.copy()

In [37]:
# Filtramos pares de SKUs con una frecuencia mayor a 10 en conjunto
final_pairs = pairs_2025[pairs_2025['FRECUENCIA']>=10]

In [39]:
final_pairs.shape

(39490, 19)

In [41]:
# Exportamos a un excel
output = 'feb_2026' # <- cambiamos el nombre a requerimiento

final_pairs.to_excel(f"C:\\Users\\fdavila\\OneDrive - Farmacorp S.A\\Escritorio\\cross selling\\output\\{output}.xlsx")